# Заняття 14 — Якість даних (Data Quality)

## Основні цілі заняття:
* Систематизувати DQ-проблеми NYC Taxi, які ми вже бачили під час EDA в занятті 02.
* Показати **три інструменти** валідації: `pandera` (code-first), `great_expectations` (suite + reporting) і `soda-core` (YAML + SQL pushdown).
* Розкласти якість даних по **п'яти осях** і навчитися класифікувати будь-яку проблему.
* Розібрати ключову інженерну дихотомію: **detective vs preventive** контроль якості (reactive ↔ proactive) — і де тут місце DQ-gate'ів.

* **Датасет:** NYC TLC Yellow Taxi Trip Records, January 2024
* **Формат:** Parquet (~100 MB, ~3 млн рядків)
* **Місток із заняття 02:** там ми робили EDA і фіксували проблеми зі словами «виправимо пізніше». Сьогодні — виправляємо систематично.

In [1]:
import json
import shutil
from datetime import datetime
from pathlib import Path

import pandas as pd
import pandera.pandas as pa
import requests
from icecream import ic

# Run from this code/ dir. Shared source; outputs go to ../../data/lesson-14/.
LANDING_DIR = Path("../../data/source")
QUARANTINE_DIR = Path("../../data/lesson-14/quarantine")
PUBLISHED_DIR = Path("../../data/lesson-14/published")
LANDING_DIR.mkdir(parents=True, exist_ok=True)
QUARANTINE_DIR.mkdir(parents=True, exist_ok=True)
PUBLISHED_DIR.mkdir(parents=True, exist_ok=True)

YEAR, MONTH = 2024, 1
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
PARQUET_PATH = LANDING_DIR / f"yellow_tripdata_{YEAR}-{MONTH:02d}.parquet"

## 1. Завантаження даних

Той самий датасет, що й у занятті 02. Скрипт ідемпотентний — повторний запуск нічого не перезаписує.

In [2]:
if not PARQUET_PATH.exists():
    url = f"{BASE_URL}/{PARQUET_PATH.name}"
    ic(url)
    r = requests.get(url, stream=True, timeout=120)
    r.raise_for_status()
    with open(PARQUET_PATH, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

df = pd.read_parquet(PARQUET_PATH)
ic(df.shape)

ic| df.shape: (2964624, 19)


(2964624, 19)

Похідні колонки `duration_h` і `speed_mph` — знадобляться для перевірки осі **Accuracy** (фізично можлива швидкість).

In [3]:
df["duration_h"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 3600
df["speed_mph"] = df["trip_distance"] / df["duration_h"].replace(0, float("nan"))
df[["tpep_pickup_datetime", "tpep_dropoff_datetime", "duration_h", "speed_mph"]].head(3)

,tpep_pickup_datetime,tpep_dropoff_datetime,duration_h,speed_mph
0,2024-01-01 00:57:55,2024-01-01 01:17:43,0.330000,5.212121
1,2024-01-01 00:03:00,2024-01-01 00:09:36,0.110000,16.363636
2,2024-01-01 00:17:06,2024-01-01 00:35:01,0.298611,15.739535


## 2. П'ять осей якості даних

«Якість даних» — надто розмите поняття, щоб ним керувати. Тому його розкладають на **виміри (dimensions)**.
Класична п'ятірка (DAMA-DMBOK, FDE):

| Вісь | Питання, на яке відповідає | NYC Taxi приклад |
|---|---|---|
| **Completeness** | Чи присутні всі очікувані значення? | NULL `passenger_count` |
| **Validity** | Чи значення в допустимому домені/діапазоні? | `fare_amount < 0`, `PULocationID > 265` |
| **Consistency** | Чи поля узгоджені між собою? | `total_amount ≠ fare + tip + tolls + ...` |
| **Timeliness** | Чи дані свіжі й у правильному часовому вікні? | `pickup_datetime` поза січнем 2024 |
| **Accuracy** | Чи значення відповідає реальності? | швидкість > 200 mph — фізично неможливо |

Часто додають **шосту** вісь — **Uniqueness** (відсутність дублікатів за ключем). Вона з'являється не в самому файлі, а під час re-ingestion, тому ми винесемо її в розділ 6 (DQ-gate).

Головне, чого треба навчитися: не зазубрити список, а вміти будь-яку конкретну проблему **покласти на одну з осей**.

### Completeness — відсутні значення

In [4]:
nulls = df.isnull().sum()
nulls[nulls > 0]

passenger_count         140162
RatecodeID              140162
store_and_fwd_flag      140162
congestion_surcharge    140162
Airport_fee             140162
speed_mph                  814
dtype: int64

### Validity — значення поза допустимим доменом

In [5]:
neg_fare = (df["fare_amount"] < 0).sum()
bad_location = ((df["PULocationID"] < 1) | (df["PULocationID"] > 265)).sum()
bad_pax = ((df["passenger_count"] < 1) | (df["passenger_count"] > 9)).sum()
ic(neg_fare, bad_location, bad_pax)

ic| neg_fare: np.int64(37448)
    bad_location: np.int64(0)
    bad_pax: np.int64(31465)


(np.int64(37448), np.int64(0), np.int64(31465))

### Consistency — поля не узгоджені між собою

`total_amount` має дорівнювати сумі компонентів. Шукаємо рядки, де рівність порушена більш ніж на 1 цент.

In [6]:
components = (
    df["fare_amount"] + df["extra"] + df["mta_tax"] + df["tip_amount"]
    + df["tolls_amount"] + df["improvement_surcharge"]
    + df["congestion_surcharge"].fillna(0) + df["Airport_fee"].fillna(0)
)
inconsistent = (df["total_amount"] - components).abs() > 0.01
ic(int(inconsistent.sum()))

ic| int(inconsistent.sum()): 761838


761838

### Timeliness — дані поза очікуваним часовим вікном

Файл — за січень 2024. Усе, що поза цим вікном, — підозріле (з минулого або майбутнього).

In [7]:
ic(df["tpep_pickup_datetime"].min(), df["tpep_pickup_datetime"].max())
out_of_window = (
    (df["tpep_pickup_datetime"] < "2024-01-01") | (df["tpep_pickup_datetime"] > "2024-02-01")
).sum()
ic(int(out_of_window))

ic| df["tpep_pickup_datetime"].min(): Timestamp('2002-12-31 22:59:39')
    df["tpep_pickup_datetime"].max(): Timestamp('2024-02-01 00:01:15')
ic| int(out_of_window): 18


18

### Accuracy — значення суперечить реальності

Швидкість > 200 mph у Нью-Йорку неможлива. Тип `float` це дозволяє, а реальність — ні.

In [8]:
impossible_speed = (df["speed_mph"] > 70).sum()
ic(int(impossible_speed))

ic| int(impossible_speed): 1192


1192

### DQ-профіль: усі осі в одній таблиці

Зведена таблиця — це вже маленький **DQ scorecard**: скільки рядків порушує кожне правило і який це відсоток.

In [9]:
total = len(df)
profile = pd.DataFrame(
    [
        ("Completeness", "passenger_count NOT NULL", int(df["passenger_count"].isna().sum())),
        ("Validity", "fare_amount >= 0", int(neg_fare)),
        ("Validity", "PULocationID in 1..265", int(bad_location)),
        ("Consistency", "total_amount = sum(components)", int(inconsistent.sum())),
        ("Timeliness", "pickup in Jan 2024", int(out_of_window)),
        ("Accuracy", "speed_mph <= 200", int(impossible_speed)),
    ],
    columns=["axis", "rule", "violations"],
)
profile["pct"] = (profile["violations"] / total * 100).round(3)
profile

,axis,rule,violations,pct
0,Completeness,passenger_count NOT NULL,140162,4.728
1,Validity,fare_amount >= 0,37448,1.263
2,Validity,PULocationID in 1..265,0,0.000
3,Consistency,total_amount = sum(components),761838,25.698
4,Timeliness,pickup in Jan 2024,18,0.001
5,Accuracy,speed_mph <= 200,1192,0.040


## 3. Pandera — декларативна схема

Pandera описує **очікувану** схему як Python-клас і валідує DataFrame проти неї.
Code-first підхід: схема живе поряд із кодом пайплайну, її легко покрити юніт-тестами і запускати в CI/CD.

- `pa.Field(ge=0)` — обмеження на значення (`ge` = greater-or-equal).
- `nullable=True` — колонка може містити NULL.
- `coerce=True` — Pandera спробує привести типи перед перевіркою.

Кожне поле схеми — це фактично одна з осей якості.

In [10]:
class TaxiTripSchema(pa.DataFrameModel):
    # Completeness + Validity
    fare_amount: float = pa.Field(ge=0, nullable=False)
    trip_distance: float = pa.Field(ge=0)
    passenger_count: float = pa.Field(ge=1, le=9, nullable=True)
    # Validity (домен taxi-зон)
    PULocationID: int = pa.Field(ge=1, le=265)
    DOLocationID: int = pa.Field(ge=1, le=265)
    # Timeliness
    tpep_pickup_datetime: pd.Timestamp = pa.Field(
        ge=pd.Timestamp("2024-01-01"), lt=pd.Timestamp("2024-02-01")
    )

    class Config:
        coerce = True

### Custom check — вісь Consistency

Діапазонів колонок недостатньо для cross-column правил. Для осі Consistency пишемо власний `@pa.dataframe_check`.

In [11]:
class TaxiTripStrictSchema(TaxiTripSchema):
    @pa.dataframe_check
    def total_amount_is_consistent(cls, df: pd.DataFrame) -> pd.Series:
        components = (
            df["fare_amount"] + df["extra"] + df["mta_tax"] + df["tip_amount"]
            + df["tolls_amount"] + df["improvement_surcharge"]
            + df["congestion_surcharge"].fillna(0) + df["Airport_fee"].fillna(0)
        )
        return (df["total_amount"] - components).abs() <= 0.01

### `lazy=True` — зібрати ВСІ помилки одразу

За замовчуванням Pandera падає на першій помилці. `lazy=True` проходить усі перевірки і повертає повний звіт `failure_cases` — саме те, що потрібно для DQ-репорту.

Валідуємо на семплі (валідація 3 млн рядків з усіма check'ами — недешева).

In [12]:
sample = df.sample(50_000, random_state=42)

try:
    TaxiTripStrictSchema.validate(sample, lazy=True)
    print("Schema OK")
except pa.errors.SchemaErrors as exc:
    failure_cases = exc.failure_cases
    dir(exc)

In [13]:
failure_cases

,schema_context,column,check,check_number,failure_case,index
132146,DataFrameSchema,fare_amount,total_amount_is_consistent,0,10.0,2711195
176217,DataFrameSchema,tip_amount,total_amount_is_consistent,0,3.5,2077117
176187,DataFrameSchema,tip_amount,total_amount_is_consistent,0,1.0,2398398
176188,DataFrameSchema,tip_amount,total_amount_is_consistent,0,2.75,1568776
176189,DataFrameSchema,tip_amount,total_amount_is_consistent,0,5.8,1787909
...,...,...,...,...,...,...
833,Column,passenger_count,greater_than_or_equal_to(1),0,0.0,1892047
834,Column,passenger_count,greater_than_or_equal_to(1),0,0.0,1534191
835,Column,passenger_count,greater_than_or_equal_to(1),0,0.0,359985
836,Column,passenger_count,greater_than_or_equal_to(1),0,0.0,533912


`failure_cases` — це DataFrame: яка перевірка впала, яке значення, в якому рядку. Згрупуємо по перевірці:

In [14]:
failure_cases.groupby("check").size().sort_values(ascending=False)

check
total_amount_is_consistent                       263045
greater_than_or_equal_to(0)                         672
greater_than_or_equal_to(1)                         575
greater_than_or_equal_to(2024-01-01 00:00:00)         1
dtype: int64

### Pandera як фільтр-трансформація

Окрім «валідуй або кидай помилку», Pandera вміє повертати **тільки валідні рядки** через `drop_invalid_rows=True` в `Config` (вимагає `lazy=True`).
Це вже місток до pipeline-патернів розділу 6 — відокремлення хороших рядків від поганих.

In [15]:
class TaxiTripCleaner(TaxiTripSchema):
    class Config:
        coerce = True
        drop_invalid_rows = True


clean = TaxiTripCleaner.validate(sample, lazy=True)
ic(len(sample), len(clean), len(sample) - len(clean))

ic| len(sample): 50000
    len(clean): 48752
    len(sample) - len(clean): 1248


(50000, 48752, 1248)

## 4. Great Expectations — expectation suite

Great Expectations (GX) — це той самий принцип декларативних правил, але з прицілом на **командну роботу і звітність**:

- **Expectation** — одне правило («колонка X не містить NULL»). Понад 50 вбудованих типів.
- **Expectation Suite** — набір правил для датасету.
- **Validator / Batch** — виконує suite проти конкретних даних.
- **Data Docs** — згенерований HTML-звіт, зрозумілий навіть менеджеру.

> Зверни увагу: API GX змінювався кілька разів. Тут — сучасний **GX 1.x Fluent API** (`context.data_sources...`), а не застарілий `RuntimeBatchRequest`, який ще зустрічається в старих туторіалах.

In [16]:
import great_expectations as gx
from great_expectations.data_context.types.base import ProgressBarsConfig

context = gx.get_context()  # ephemeral in-memory context
context.variables.progress_bars = ProgressBarsConfig(globally=False)  # без tqdm-шуму

data_source = context.data_sources.add_pandas("taxi_pandas")
data_asset = data_source.add_dataframe_asset("yellow_trips")
batch_definition = data_asset.add_batch_definition_whole_dataframe("batch")
batch = batch_definition.get_batch(batch_parameters={"dataframe": sample})

### Будуємо suite — по одній expectation на вісь якості

In [17]:
suite = gx.ExpectationSuite("silver_yellow_trips")

# Completeness
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="fare_amount"))
# Validity
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(column="fare_amount", min_value=0)
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="PULocationID", min_value=1, max_value=265
    )
)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column="passenger_count", min_value=1, max_value=9
    )
)
# Accuracy (через похідну колонку)
suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(column="speed_mph", max_value=200)
)
# Uniqueness (compound key)
suite.add_expectation(
    gx.expectations.ExpectCompoundColumnsToBeUnique(
        column_list=["tpep_pickup_datetime", "PULocationID", "DOLocationID", "fare_amount"]
    )
)

ic(len(suite.expectations))

ic| len(suite.expectations): 6


6

### Запускаємо валідацію

In [18]:
results = batch.validate(suite)
ic(results.success)
results.statistics

ic| results.success: False


{'evaluated_expectations': 6,
 'successful_expectations': 3,
 'unsuccessful_expectations': 3,
 'success_percent': 50.0}

Розкладемо результат по кожній expectation у читабельну таблицю — це GX-аналог нашого scorecard'у:

In [19]:
gx_report = pd.DataFrame(
    [
        {
            "expectation": r.expectation_config.type,
            "column": r.expectation_config.kwargs.get("column")
            or r.expectation_config.kwargs.get("column_list"),
            "success": r.success,
            "unexpected": r.result.get("unexpected_count"),
            "unexpected_pct": round(r.result.get("unexpected_percent") or 0, 3),
        }
        for r in results.results
    ]
)
gx_report

,expectation,column,success,unexpected,unexpected_pct
0,expect_column_values_to_not_be_null,fare_amount,True,0,0.000
1,expect_column_values_to_be_between,fare_amount,False,672,1.344
2,expect_column_values_to_be_between,PULocationID,True,0,0.000
3,expect_column_values_to_be_between,passenger_count,False,575,1.207
4,expect_column_values_to_be_between,speed_mph,False,13,0.026
5,expect_compound_columns_to_be_unique,"[tpep_pickup_datetime, PULocationID, DOLocatio...",True,0,0.000


### Pandera чи Great Expectations?

| | **Pandera** | **Great Expectations** |
|---|---|---|
| Стиль | схема як Python-клас | suite з декларативних expectations |
| Найкраще для | валідація в коді, CI/CD, юніт-тести | shared DQ across teams, звітність |
| Артефакт | exception + `failure_cases` DataFrame | `ValidationResult` + Data Docs (HTML) |
| Поріг входу | низький (один декоратор) | вищий (context, datasource, suite) |
| Коли брати | бібліотека / dbt-стиль пайплайн | велика організація, аудит, нетехнічні стейкхолдери |

Це не «або-або»: Pandera часто стоїть на вході в код (швидка gate-перевірка), а GX — на рівні shared таблиць зі звітами для всіх команд.

## 5. Soda Core — YAML-чеки з SQL pushdown

Pandera і GX виконують перевірки **в Python-процесі**: дані спершу треба підняти в пам'ять як DataFrame.
Soda влаштована інакше — правила описані в **YAML (SodaCL)**, а сам чек компілюється в **SQL і виконується у сховищі**.
Дані нікуди не їдуть, назад повертаються лише метрики. Для таблиці на мільярд рядків це різниця між «неможливо» і «кілька секунд».

| | Pandera / GX | Soda |
|---|---|---|
| Де живе правило | Python-код | YAML-файл |
| Де рахується | у процесі, над DataFrame | у сховищі, SQL pushdown |
| Хто редагує | інженер | інженер і аналітик |
| Що йде по мережі | увесь batch | лише метрики |

Перевірятимемо **ті самі осі й ті самі пороги**, що й у Pandera/GX — щоб різниця була саме в підході, а не в правилах.

### Крок 1 — покласти дані у сховище

Soda перевіряє **таблицю**, а не DataFrame. Тому спершу матеріалізуємо наш `sample` у DuckDB (те саме сховище, що в занятті 04).

In [20]:
import subprocess

import duckdb

SODA_DIR = Path("soda")
SODA_DIR.mkdir(exist_ok=True)
DUCKDB_PATH = (Path("../../data/lesson-14") / "dq.duckdb").resolve()

con = duckdb.connect(str(DUCKDB_PATH))
con.execute("CREATE OR REPLACE TABLE silver_yellow_trips AS SELECT * FROM sample")
ic(con.execute("SELECT count(*) FROM silver_yellow_trips").fetchone()[0])
con.close()

ic| con.execute("SELECT count(*) FROM silver_yellow_trips").fetchone()[0]: 50000


### Крок 2 — `configuration.yml`: до чого підключатися

Один файл на джерело даних. У проді тут був би Snowflake/BigQuery/Postgres — сам SodaCL від цього не змінюється.

In [21]:
(SODA_DIR / "configuration.yml").write_text(f"""\
data_source taxi:
  type: duckdb
  path: {DUCKDB_PATH}
  read_only: true
""")
print((SODA_DIR / "configuration.yml").read_text())

data_source taxi:
  type: duckdb
  path: /Users/illia/projects/UA_DATA-ENGINEERING_KHOROSHYKH-2-hw/data/lesson-14/dq.duckdb
  read_only: true



### Крок 3 — `checks.yml`: правила мовою SodaCL

Ключові конструкції:

- `missing_count` / `missing_percent` — вісь **Completeness**;
- `invalid_count` / `invalid_percent` + блок `valid min` / `valid max` / `valid values` — вісь **Validity**;
- `duplicate_count(a, b, c)` — вісь **Uniqueness** по складеному ключу;
- `warn: when > 2 %` / `fail: when > 5 %` — **вбудовані пороги з двома рівнями серйозності**. Це аналог `mostly` у GX, але з окремим станом WARN, якого в GX немає;
- `failed rows` з `fail query` — довільний SQL для cross-column правил (вісь **Consistency**), аналог `@pa.dataframe_check` у Pandera;
- користувацька метрика (`out_of_window query`) — коли вбудованих метрик не вистачає (вісь **Timeliness**: `valid min`/`valid max` у Soda 3.x приймають лише числа, не дати).

### Крок 4 — запустити скан

Soda — це передусім **CLI**: `soda scan -d <джерело> -c configuration.yml checks.yml`. Так її і запускають в Airflow (`BashOperator`) чи в CI.

> **Чому через `uv run --no-project`, а не звичайний import?**
> `soda-core` тягне `minimal-snowplow-tracker`, який займає той самий модуль `snowplow_tracker`, що й `dbt-core`. В одному venv вони не співіснують — поставивши Soda в основне оточення, ми зламали б dbt із занять 08 і 13.
> Тому Soda живе в **ізольованому ephemeral-оточенні**. Це не костур для ноутбука, а нормальна практика: DQ-інструмент ставлять окремим контейнером/venv від самого пайплайну.

In [22]:
scan_cmd = [
    "uv", "run", "--no-project",
    "--with", "soda-core-duckdb==3.3.20", "--with", "duckdb>=1.5.5",
    "--", "soda", "scan", "-d", "taxi",
    "-c", "configuration.yml", "checks.yml",
    "--scan-results-file", "scan_results.json",
]
scan = subprocess.run(scan_cmd, cwd=SODA_DIR, capture_output=True, text=True)
print(scan.stdout[scan.stdout.find("Scan summary:"):])

Scan summary:
[21:54:39] 5/9 checks PASSED: 
[21:54:39]     silver_yellow_trips in taxi
[21:54:39]       row_count > 0 [PASSED]
[21:54:39]       duplicate_count(tpep_pickup_datetime, PULocationID, DOLocationID, fare_amount) = 0 [PASSED]
[21:54:39]       missing_count(fare_amount) = 0 [PASSED]
[21:54:39]       invalid_count(PULocationID) = 0 [PASSED]
[21:54:39]       invalid_percent(passenger_count) warn when > 2 % fail when > 5 % [PASSED]
[21:54:39] 4/9 checks FAILED: 
[21:54:39]     silver_yellow_trips in taxi
[21:54:39]       out_of_window = 0 [FAILED]
[21:54:39]         check_value: 1.0
[21:54:39]       total_amount = сума компонентів [FAILED]
[21:54:39]         value: 13035
[21:54:39]       invalid_count(fare_amount) = 0 [FAILED]
[21:54:39]         check_value: 672
[21:54:39]       invalid_count(speed_mph) = 0 [FAILED]
[21:54:39]         check_value: 13
[21:54:39] Oops! 4 failures. 0 warnings. 0 errors. 5 pass.
Saving scan results to scan_results.json



**Exit code — це і є інтерфейс для оркестратора**: `0` — усі чеки пройшли, `2` — є FAIL, `3` — є WARN або помилка.
Саме на ньому будується gate: `BashOperator` з `soda scan` впаде сам, без жодного Python-коду. Це місток до розділу 6.

In [23]:
ic(scan.returncode)

ic| scan.returncode: 2


2

### Крок 5 — результат як таблиця

`--scan-results-file` дає машиночитний JSON — те, з чого будують дашборд або відправляють у Soda Cloud. Зберемо з нього scorecard, аналогічний `gx_report`:

In [24]:
scan_results = json.loads((SODA_DIR / "scan_results.json").read_text())
soda_report = pd.DataFrame(
    [
        {
            "check": c["name"],
            "column": c["column"],
            "outcome": c["outcome"],
            "value": c["diagnostics"].get("value"),
        }
        for c in scan_results["checks"]
    ]
)
soda_report

,check,column,outcome,value
0,row_count > 0,None,pass,50000.00
1,out_of_window = 0,None,fail,1.00
2,"duplicate_count(tpep_pickup_datetime, PULocati...",None,pass,0.00
3,total_amount = сума компонентів,None,fail,13035.00
4,missing_count(fare_amount) = 0,fare_amount,pass,0.00
5,invalid_count(fare_amount) = 0,fare_amount,fail,672.00
6,invalid_count(PULocationID) = 0,PULocationID,pass,0.00
7,invalid_percent(passenger_count) warn when > 2...,passenger_count,pass,1.15
8,invalid_count(speed_mph) = 0,speed_mph,fail,13.00


Знахідки збіглися з Pandera і GX — від'ємні `fare_amount`, неможливі швидкості, розсинхрон `total_amount` — але жоден рядок не залишав DuckDB: усе порахував SQL.

### Pandera, GX чи Soda?

| | **Pandera** | **Great Expectations** | **Soda Core** |
|---|---|---|---|
| Форма опису | Python-клас | Python-suite | YAML (SodaCL) |
| Де виконується | у процесі, над DataFrame | у процесі або окремим кроком | у сховищі (SQL pushdown) |
| Пороги | через власні перевірки | `mostly=0.95` | `warn:` / `fail:` — два рівні |
| Звітність | `failure_cases` DataFrame | Data Docs (HTML) | JSON, дашборд Soda Cloud |
| Профілювання | ні | так | так |
| Найкраще для | валідація в коді, CI/CD | shared suites, аудит, звіти | моніторинг таблиць у сховищі |

Вони не взаємовиключні. Типова зв'язка: **Pandera** — на межах функцій у коді, **тести dbt** — на моделях сховища, **GX або Soda** — як gate між шарами й джерело звітності.

## 6. Detective vs Preventive: reactive і proactive підхід

Питання не «чи перевіряти дані», а **що робити з результатом перевірки**. Сама перевірка (Pandera/GX) — нейтральна; її поведінку визначає **те, куди ти її вмонтуєш**.

Корисно розкласти це на **дві незалежні осі** (їх часто плутають):

**Вісь A — ЯК шукаємо проблему:**
- **Assertion / testing** — явні правила, детермінований pass/fail. Pandera, GX, dbt tests, Soda Core. Ловить *known-unknowns* (проблеми, які ми передбачили).

- **Observability / monitoring** — статистичний моніторинг метрик у часі: freshness, volume, schema drift, distribution, lineage. Ловить *unknown-unknowns*. Це Monte Carlo, Soda Cloud, anomaly detection.

**Вісь B — ЩО робимо на failure (це і є reactive ↔ proactive):**
- **Detective (reactive)** — записали результат, підняли alert, **але дані пішли далі**. Про проблему дізнаємось post-factum, можливо вже після того, як її побачив споживач.
- **Preventive (proactive)** — перевірка є **блокуючим gate'ом**: failure зупиняє пайплайн, погані дані **не доходять** до наступного шару. Це **circuit breaker** / патерн **Write-Audit-Publish (WAP)**.

**Відповідь на питання «куди належать DQ-gate'и?»**: DQ-gate (quality gate / circuit breaker / WAP) — це **найчистіша форма preventive контролю**. Він не належить до «алертингу/SLA» — це окрема вісь. А «DQ як частина SLA» — це вже **організаційний шар** (data contracts, SLO, data ownership), який задає *цілі*, проти яких обидві осі вимірюються. Цей підхід називають **Data Reliability Engineering** (SRE для даних).

### Спільна перевірка для обох підходів

Винесемо валідацію в одну функцію, яка повертає чисті рядки, погані рядки і метрику pass-rate. Далі покажемо, як та сама перевірка стає або detective, або preventive — залежно від обгортки.

In [25]:
def run_dq_check(batch_df: pd.DataFrame) -> dict:
    """Detective core: валідує batch, розділяє рядки, рахує pass-rate. Нічого не блокує."""
    valid = TaxiTripCleaner.validate(batch_df, lazy=True)
    invalid_idx = batch_df.index.difference(valid.index)
    invalid = batch_df.loc[invalid_idx]
    pass_rate = len(valid) / len(batch_df) if len(batch_df) else 1.0
    return {"valid": valid, "invalid": invalid, "pass_rate": pass_rate, "rows": len(batch_df)}

### Два тестові batch'і

Щоб контраст між підходами був чесним, проженемо обидва на **тих самих даних**:
- `batch_clean` — завідомо валідні рядки (pass-rate ≈ 1.0);
- `batch_dirty` — той самий чистий набір + 5 000 рядків з від'ємним `fare_amount` (pass-rate < SLO).

In [26]:
SLO_PASS_RATE = 0.95

batch_clean = sample[
    (sample["fare_amount"] >= 0)
    & sample["PULocationID"].between(1, 265)
    & (sample["speed_mph"].fillna(0) <= 200)
].copy()

broken = sample.sample(5_000, random_state=1).copy()
broken["fare_amount"] = -broken["fare_amount"].abs()  # усі fare від'ємні
batch_dirty = pd.concat([batch_clean, broken], ignore_index=True)

ic(len(batch_clean), len(batch_dirty))

ic| len(batch_clean): 49315, len(batch_dirty): 54315


(49315, 54315)

### Підхід 1 — Detective (reactive): monitor + alert

Перевірка рахує метрику і шле alert, якщо pass-rate нижчий за поріг (SLO). Але **дані течуть далі незалежно від результату** — `detective_alert` повертає весь batch. Ти дізнаєшся про проблему, але реагуєш уже після факту.

In [27]:
def detective_alert(batch_df: pd.DataFrame) -> pd.DataFrame:
    report = run_dq_check(batch_df)
    if report["pass_rate"] < SLO_PASS_RATE:
        alert = {
            "level": "WARNING",
            "ts": datetime.now().isoformat(timespec="seconds"),
            "pass_rate": round(report["pass_rate"], 4),
            "slo": SLO_PASS_RATE,
            "bad_rows": len(report["invalid"]),
        }
        print("ALERT →", json.dumps(alert))  # у проді: Slack / PagerDuty / email
    return batch_df  # дані повертаються повністю — pipeline НЕ зупинено

flowed = detective_alert(batch_dirty)
ic(len(flowed))  # усі рядки пройшли далі, включно з поганими

ALERT → {"level": "WARNING", "ts": "2026-09-09T21:54:39", "pass_rate": 0.8974, "slo": 0.95, "bad_rows": 5573}


ic| len(flowed): 54315


54315

### Підхід 2 — Preventive (proactive): Write-Audit-Publish gate

Той самий чек, але вмонтований як **блокуючий gate**:

1. **Write** — записуємо batch у `staging` (ще не видимий споживачам).
2. **Audit** — проганяємо DQ-чек.
3. **Publish** — якщо pass-rate >= SLO: чисті рядки публікуються в `published`, погані їдуть у `quarantine`. Якщо ні: **gate кидає виняток, нічого не публікується**.

Погані дані фізично не доходять до наступного шару. Це і є DQ-gateway.

In [28]:
for d in (QUARANTINE_DIR, PUBLISHED_DIR):
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)


class DataQualityGateError(Exception):
    """Підняти = заблокувати пайплайн (circuit breaker)."""


def write_audit_publish(batch_df: pd.DataFrame, batch_name: str) -> Path:
    report = run_dq_check(batch_df)
    ic(batch_name, round(report["pass_rate"], 4))

    # bad rows завжди в карантин (не мовчки викидаємо — зберігаємо для розбору)
    if len(report["invalid"]):
        report["invalid"].to_parquet(QUARANTINE_DIR / f"{batch_name}.parquet")

    if report["pass_rate"] < SLO_PASS_RATE:
        raise DataQualityGateError(
            f"{batch_name}: pass_rate {report['pass_rate']:.4f} < SLO {SLO_PASS_RATE} — publish BLOCKED"
        )

    out = PUBLISHED_DIR / f"{batch_name}.parquet"
    report["valid"].to_parquet(out)
    return out

**Batch A — чистий**: gate пропускає, дані публікуються.

In [29]:
published_path = write_audit_publish(batch_clean, "2024-01_clean")
ic(published_path.exists(), len(pd.read_parquet(published_path)))

ic| batch_name: '2024-01_clean', round(report["pass_rate"], 4): 0.9884
ic| published_path.exists(): True
    len(pd.read_parquet(published_path)): 48742


(True, 48742)

**Batch B — зіпсований** (той самий, що detective пропустив далі): gate блокує, в `published` нічого не з'являється, биті рядки — в карантині.

In [30]:
try:
    write_audit_publish(batch_dirty, "2024-01_dirty")
except DataQualityGateError as exc:
    print("BLOCKED →", exc)

BLOCKED → 2024-01_dirty: pass_rate 0.8974 < SLO 0.95 — publish BLOCKED


ic| batch_name: '2024-01_dirty', round(report["pass_rate"], 4): 0.8974


In [31]:
ic(
    sorted(p.name for p in PUBLISHED_DIR.glob("*.parquet")),
    sorted(p.name for p in QUARANTINE_DIR.glob("*.parquet")),
)

ic| sorted(p.name for p in PUBLISHED_DIR.glob("*.parquet")): ['2024-01_clean.parquet']
    sorted(p.name for p in QUARANTINE_DIR.glob("*.parquet")): ['2024-01_clean.parquet', '2024-01_dirty.parquet']


(['2024-01_clean.parquet'], ['2024-01_clean.parquet', '2024-01_dirty.parquet'])

Результат: у `published` — лише чистий batch; зіпсований навіть не дійшов, його сліди — в `quarantine` для розбору. У detective-підході обидва batch'і пройшли б далі.

### Вісь A на практиці: observability (unknown-unknowns)

Assertion-перевірки ловлять те, що ми передбачили. Але дані «ламаються» і способами, які ми не закодували в правилах: труба не відпрацювала (freshness), джерело віддало вдвічі менше рядків (volume), денна виручка стрибнула (distribution). Це територія **observability** — моніторинг метрик, а не правил.

In [32]:
# Freshness: наскільки старі найсвіжіші дані?
latest = df["tpep_pickup_datetime"].max()
ic(latest, datetime(2024, 2, 1) - latest.to_pydatetime())

# Distribution: денна виручка, що відхиляється > 3σ від середнього (anomaly detection)
daily = (
    df.assign(day=df["tpep_pickup_datetime"].dt.floor("D"))
    .groupby("day")["total_amount"].sum()
)
in_window = daily[(daily.index >= "2024-01-01") & (daily.index < "2024-02-01")]
anomalies = in_window[(in_window - in_window.mean()).abs() > 3 * in_window.std()]
anomalies

ic| latest: Timestamp('2024-02-01 00:01:15')
    datetime(2024, 2, 1) - latest.to_pydatetime(): datetime.timedelta(days=-1, seconds=86325)


Series([], Name: total_amount, dtype: float64)

## 7. Підсумок: ландшафт інструментів

**Escalation spiral якості даних у нашому курсі** — одна ідея, що поглиблюється з заняття в заняття:

1. **Заняття 02** — `pandera`: локальна schema-валідація, перші DQ-спостереження.
2. **Заняття 03 / 13** — `dbt tests` (`not_null`, `unique`, `accepted_values`): перевірки на рівні пайплайну.
3. **Заняття 14 (сьогодні)** — `great_expectations` і `soda-core`: повні expectation suites, чеки у сховищі + WAP-gate.

**Ширший ландшафт** (що ще існує і коли брати):

| Інструмент | Ніша |
|---|---|
| **Pandera** | code-first валідація pandas/polars/Spark, CI/CD |
| **dbt tests + dbt-utils** | SQL-перевірки всередині dbt-моделей |
| **Great Expectations** | shared suites + Data Docs для команд |
| **Soda Core** | YAML-чеки з SQL pushdown — золота середина |
| **Elementary** | dbt-native observability, дешево й open-source |
| **PyDeequ / Deequ** | DQ нативно у Spark (AWS-екосистема) |
| **Monte Carlo / Bigeye** | повноцінна data observability (платні) |

**Що винести з заняття:**
- Якість даних = **5 (або 6) осей** + **інструмент перевірки** + **місце вмонтування** (detective чи preventive) + **організаційний контракт** (SLA / ownership).
- DQ-gate (WAP) — це інженерний контроль, який перетворює «ми помітили проблему» на «погані дані не пройшли».
- Биті рядки **карантинимо**, а не викидаємо мовчки — інакше втрачаємо можливість розслідувати.